# Pipe Endpoint Detection by Voxel Graph

This notebook visualizes pipe end candidates detected from a coarse voxel graph. It is separate from the cylinder segmentation notebook.

In [ ]:
from pathlib import Path
import sys
import json
import importlib

import numpy as np
import open3d as o3d

PLUGIN_ROOT = Path.cwd()
if PLUGIN_ROOT.name != "poseDeterminator":
    PLUGIN_ROOT = Path("python/plugins/poseDeterminator").resolve()

if str(PLUGIN_ROOT) not in sys.path:
    sys.path.insert(0, str(PLUGIN_ROOT))

import PipeEndProfileAnalyzer as pipe_analyzer_module
pipe_analyzer_module = importlib.reload(pipe_analyzer_module)
analyze_pipe_endpoints_by_voxel_graph = pipe_analyzer_module.analyze_pipe_endpoints_by_voxel_graph

PLUGIN_ROOT


## Settings

`GRAPH_VOXEL_SIZE` is the main parameter. Use `None` to auto-pick a value from the pipe bounding box. If the graph is disconnected, increase it. If the endpoint is too coarse, decrease it.

In [ ]:
PIPE_PATH = PLUGIN_ROOT / "data" / "PIPE NO.1.ply"
# PIPE_PATH = PLUGIN_ROOT / "data" / "PIPE NO.3_fill.ply"

SCALE = 1.0
MAX_POINTS = 50000

GRAPH_VOXEL_SIZE = None  # auto; for mm-scale samples try 40~100, for m-scale samples try 0.04~0.10
GRAPH_MIN_VOXEL_POINTS = 2
GRAPH_NEIGHBOR_RADIUS = 1

VISUAL_MAX_POINTS = 20000

assert PIPE_PATH.exists(), PIPE_PATH
PIPE_PATH


## Run Detector

The detector builds a voxel graph, keeps the largest connected component, and selects the graph-diameter endpoints.

In [ ]:
pipe_analyzer_module = importlib.reload(pipe_analyzer_module)
analyze_pipe_endpoints_by_voxel_graph = pipe_analyzer_module.analyze_pipe_endpoints_by_voxel_graph

graph_result = analyze_pipe_endpoints_by_voxel_graph(
    PIPE_PATH,
    scale=SCALE,
    voxel_size=GRAPH_VOXEL_SIZE,
    max_points=MAX_POINTS,
    min_voxel_points=GRAPH_MIN_VOXEL_POINTS,
    neighbor_radius_voxels=GRAPH_NEIGHBOR_RADIUS,
    include_points=False,
    random_seed=0,
)

summary = {
    "voxel_size": graph_result["voxel_size"],
    "point_count": graph_result["point_count"],
    "fit_point_count": graph_result["fit_point_count"],
    "node_count": graph_result["node_count"],
    "component_node_count": graph_result["component_node_count"],
    "edge_count": graph_result["edge_count"],
    "diameter_distance": graph_result["diameter_distance"],
    "terminal_node_indices": graph_result["terminal_node_indices"],
    "terminal_end_points": graph_result["terminal_end_points"],
    "timing_sec": graph_result["timing_sec"],
}
print(json.dumps(summary, indent=2, ensure_ascii=False))


## Visualize

Gray: original pipe points. Blue: voxel graph. Yellow: graph-diameter path. Red/green: detected pipe ends.

In [ ]:
def make_lines(points, lines, color):
    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(np.asarray(points, dtype=float))
    line_set.lines = o3d.utility.Vector2iVector(np.asarray(lines, dtype=np.int32))
    line_set.paint_uniform_color(color)
    return line_set


context_pcd = o3d.io.read_point_cloud(str(PIPE_PATH))
if SCALE != 1.0:
    context_pcd.scale(SCALE, center=np.zeros(3))
if len(context_pcd.points) > VISUAL_MAX_POINTS:
    context_pcd = context_pcd.random_down_sample(VISUAL_MAX_POINTS / len(context_pcd.points))
context_pcd.paint_uniform_color([0.72, 0.72, 0.72])

nodes = np.asarray(graph_result["graph"]["nodes"], dtype=float)
edges = graph_result["graph"]["edges"]
path_node_indices = graph_result["graph"]["path_node_indices"]
terminal_points = np.asarray(graph_result["terminal_end_points"], dtype=float)

node_pcd = o3d.geometry.PointCloud()
node_pcd.points = o3d.utility.Vector3dVector(nodes)
node_pcd.paint_uniform_color([0.0, 0.2, 1.0])

bbox_extent = context_pcd.get_axis_aligned_bounding_box().get_extent()
effective_voxel_size = float(graph_result["voxel_size"])
marker_radius = max(float(np.linalg.norm(bbox_extent)) * 0.012, effective_voxel_size * 0.35)

geometries = [context_pcd, node_pcd]
if edges:
    geometries.append(make_lines(nodes, edges, [0.0, 0.45, 1.0]))

if len(path_node_indices) >= 2:
    path_points = nodes[path_node_indices]
    path_lines = [[i, i + 1] for i in range(len(path_points) - 1)]
    geometries.append(make_lines(path_points, path_lines, [1.0, 0.85, 0.0]))

endpoint_colors = [[1.0, 0.0, 0.0], [0.0, 0.8, 0.2]]
for idx, point in enumerate(terminal_points):
    sphere = o3d.geometry.TriangleMesh.create_sphere(radius=marker_radius, resolution=24)
    sphere.translate(point)
    sphere.paint_uniform_color(endpoint_colors[idx % len(endpoint_colors)])
    geometries.append(sphere)

o3d.visualization.draw_geometries(geometries)
